In [ ]:
!pip install -q google-genai pypdf sentence-transformers chromadb langchain-text-splitters

In [ ]:

import os
import chromadb

from google.colab import files, userdata
from google import genai

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
api_key = userdata.get("Gemini_API_Key_2")

client = genai.Client(api_key=api_key)

print("Gemini connected successfully")

Gemini connected successfully


In [ ]:
uploaded = files.upload()

pdf_name = list(uploaded.keys())[0]

print("Uploaded:", pdf_name)

Saving Biotechnology Research Laboratory.pdf to Biotechnology Research Laboratory.pdf
Uploaded: Biotechnology Research Laboratory.pdf


In [ ]:
reader = PdfReader(pdf_name)

text = ""

for page in reader.pages:
    text += page.extract_text()

print(text[:2000])

Biotechnology Research Laboratory 
Daily Research Log 
Date: 10 September 2026 
Researcher: Rahul Sharma 
Project: Microbial Growth Analysis 
1. Objective 
The objective of today's experiment was to study the growth of microorganisms under different 
temperature conditions. 
2. Experiment Performed 
Three samples were prepared and kept at different temperatures: 
• Sample A: 25°C 
• Sample B: 30°C 
• Sample C: 37°C 
The samples were observed at regular intervals during the experiment. 
3. Observations 
Sample A showed slow microbial growth. 
Sample B showed moderate growth. 
Sample C showed faster growth compared with the other two samples. 
The growth of microorganisms appeared to increase with temperature within the tested range. 
4. Problems Observed 
The first observation of Sample B was delayed because of an equipment issue. 
Some readings were slightly different from the expected values. 
The laboratory temperature also changed slightly during the experiment. 
5. Results 
Sample 

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_text(text)

print("Number of chunks:", len(chunks))

Number of chunks: 4


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print("Embeddings created")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings created


In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="biolab_research"
)

collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=[str(i) for i in range(len(chunks))]
)

print("Research data stored in ChromaDB")

Research data stored in ChromaDB


In [ ]:
def search_research(query):

    query_embedding = model.encode([query])[0]

    result = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=3
    )

    return result["documents"][0]

In [ ]:
def ask_gemini(question):

    relevant_text = search_research(question)

    context = "\n".join(relevant_text)

    prompt = f"""
You are a Biotechnology Lab Supervisor AI.

Use only the research log information given below.

Research Log:
{context}

Question:
{question}

Give a short and simple answer.
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

In [ ]:
question = "Give me a summary of today's research work."

answer = ask_gemini(question)

question = "What experiments were performed?"

print(ask_gemini(question))

print(answer)

Based on the provided research log, there is no mention of any specific experiments being performed. The text only includes instructions regarding literature searching, equipment calibrations, material lots, and waste tracking.
Based on the provided document, today's research work outlines the required elements for a **Research Log** assignment in the course *"3.093 – Information Exploration: Becoming a Savvy Scholar."* 

**Summary of Required Log Elements:**
* **Sources Used & Justification:** Names of sources (databases, websites, books) and specific reasons for selecting them.
* **Search Process:** Detailed steps, keywords used, and search methods.
* **Results & Selection:** Evaluation of findings (academic relevance, expected vs. actual results) and why specific results were chosen.
* **Time Spent:** Duration spent searching each source.
* **Emotional Response:** Personal feelings during the process (e.g., confident, frustrated, confused).

*Note: Research logs account for 30% of t